# Logits Masking

Logits Masking provides a way for application clients of a
foundational model to ensure that the text the model
generates conforms to a set of rules.

## Problem
Sometimes, when you’re generating text using LLMs, you
want it to conform to specific style rules. These rules might
be in place for branding, accuracy, compliance, or other
reasons.

## Solution -> Logits Masking:

#### Beam Search:
Models don’t choose continuation tokens one at a time.
Instead, they employ beam search, which is a deterministic
search algorithm that aims to find the most likely sequence
of tokens by exploring multiple possible continuations in
parallel. In this way, the model can consider the probability
of the entire sequence, not just the next token.

#### Logits Masking:
The idea behind Logits Masking is to
intercept the generation at this sampling stage. 

Logits Masking works as follows:
* Rather than wait until the full content is generated,
you obtain the set of possible continuations at each
intermediate point.
* You zero out the probability associated with
continuations that do not meet the rules.
* As long as there is at least one continuation that
meets the rules, generation can proceed.
* If there is no continuation that meets the rules or if
the generation is at a point that you have previously encountered as a dead end, you need to back up one
step and retry generation.
* After some maximum number of generation
attempts, you send a refusal back to the user saying
that you are unable to generate content that meets
the rules.

In [4]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

with open("banned_phrases.txt") as ifp:
    banned_phrases = [line.strip().lower() for line in ifp.readlines()]

with open("desired_phrases.txt") as ifp:
    desired_phrases = [line.strip().lower() for line in ifp.readlines()]

banned_phrases = set[str](banned_phrases)
desired_phrases = set[str](desired_phrases)      

from transformers import pipeline

pipe = pipeline(
    task="text-generation", 
    model=model_id,
    use_fast=True,
    kwargs={
        "return_full_text": False,
    },
    model_kwargs={}
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

### Zero shot testing

In [5]:
def generate_product_description(item: str) -> str:
    system_prompt = f"""
        You are a product marketer for a company that makes nutrition supplements.
        Balance your product descriptions to attract customers, optimize SEO, and
        stay within accurate advertising guidelines.
        Product descriptions have to be 3-5 sentences.
        Provide only the product description with no preamble.
    """
    user_prompt = f"""
        Write a product description for a {item}.
    """

    input_message = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}   
    ]
    
    results = pipe(input_message, 
                   max_new_tokens=512)
    return results[0]['generated_text'][-1]['content'].strip()

prod = generate_product_description("protein drink")
print(prod)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Introducing the amazing Protein Drink, a nutritious and delicious drink that's packed with protein to help you build and repair your muscles. Made with the finest natural ingredients, this drink provides the essential amino acids your body needs to fuel your workouts and grow strong.

Powerful blend of Whey Protein Isolate and Pea Protein, with added superfoods like Chia Seeds, Hemp Seeds, and Acai Berries for a nutrient-dense and satisfying meal replacement.

Enjoy a delicious boost of energy and endurance during your workouts with every sip, thanks to the electrolytes and minerals that will keep you hydrated and hydrated.

Protein Drink is available in 5 delicious flavors, each with a unique blend of vitamins and minerals to keep you feeling full and satisfied.

So, what are you waiting for? Order yours today and start building your muscles with the power of Protein Drink! 💪🏼


In [6]:
import numpy as np

def evaluate(descr: str, positives, negatives) -> int:
    # go through and count the number of desired phrases and banned phrases
    descr = descr.lower()
    num_positive = np.sum([1 for phrase in positives if phrase in descr])
    num_negative = np.sum([1 for phrase in negatives if phrase in descr])
    return int(num_positive - num_negative)

def evaluate_verbose(descr: str, positives, negatives) -> int:
    # go through and count the number of desired phrases and banned phrases
    descr = descr.lower()
    
    num_positive = num_negative = 0
    for phrase in positives:
        if phrase in descr:
            num_positive += 1
            print(f"Good: {phrase}")
    for phrase in negatives:
        if phrase in descr:
            num_negative += 1
            print(f"Bad: {phrase}")
    print(f"Good: {num_positive}   Bad: {num_negative}")
    return num_positive - num_negative

evaluate_verbose(prod, desired_phrases, banned_phrases)

Good: whey protein
Good: whey
Bad: star
Bad: finest
Bad: natural
Bad: add
Good: 2   Bad: 4


-2

### Using logits processing to enhance the product description

In [8]:
import torch
import numpy as np
from transformers.generation.logits_process import (
    LogitsProcessor,
    LOGITS_PROCESSOR_INPUTS_DOCSTRING,
)
from transformers.utils import add_start_docstrings

class BrandLogitsProcessor(LogitsProcessor):
    def __init__(self, tokenizer, positives, negatives):
        self.tokenizer = tokenizer
        self.positives = positives
        self.negatives = negatives
      
    @add_start_docstrings(LOGITS_PROCESSOR_INPUTS_DOCSTRING)
    def __call__(
        self, input_ids: torch.LongTensor, input_logits: torch.FloatTensor
    ) -> torch.FloatTensor:
        output_logits = input_logits.clone()
            
        num_matches = [0] * len(input_ids)
        for idx, seq in enumerate(input_ids):
            # decode the sequence
            decoded = self.tokenizer.decode(seq)
            num_matches[idx] = evaluate(decoded, self.positives, self.negatives)
        max_matches = np.max(num_matches)
          
        # logits goes from -inf to zero.  Mask out the non-max sequences; torch doesn't like it to be -np.inf
        for idx in range(len(input_ids)):
            if num_matches[idx] != max_matches:
                output_logits[idx] = -10000
                  
        return output_logits

In [10]:
def generate_product_description_v2(item: str) -> str:
    system_prompt = f"""
        You are a product marketer for a company that makes nutrition supplements.
        Balance your product descriptions to attract customers, optimize SEO, and
        stay within accurate advertising guidelines.
        Product descriptions have to be 3-5 sentences.
        Provide only the product description with no preamble.
    """
    user_prompt = f"""
        Write a product description for a {item}.
    """
    
    input_message = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}   
    ]
    
    # alliterate on the first letter of the animal. So, donkey would be D
    brand_processor = BrandLogitsProcessor(pipe.tokenizer, desired_phrases, banned_phrases)
    
    results = pipe(input_message, 
                   max_new_tokens=512,
                   do_sample=True,
                   temperature=0.8,
                   num_beams=10,
                   use_cache=True, # default is True
                   logits_processor=[brand_processor])
    return results[0]['generated_text'][-1]['content'].strip()

prod = generate_product_description_v2("protein drink")
print(prod)

evaluate_verbose(prod, desired_phrases, banned_phrases)

Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Introducing our delicious and nutritious Protein Shake:

Ingredients:
- 2 scoops of your favorite protein powder
- 1 cup of almond milk
- 1 banana, sliced
- 1 tablespoon of chia seeds
- 1 tablespoon of honey (optional)

Directions:
1. In a blender, combine the protein powder, almond milk, sliced banana, chia seeds, and honey (if using).
2. Blend until smooth and creamy.
3. Pour the mixture into a glass and enjoy your delicious and nutritious Protein Shake!

Product Description:
Our Protein Shake is packed with protein, fiber, and other essential nutrients to help you feel full and energized throughout the day. It's also gluten-free, dairy-free, and vegan-friendly, making it a great option for those with dietary restrictions.

Key Benefits:
- High protein content: Our Protein Shake contains 20-25 grams of protein per serving, making it a great option for those looking to boost their protein intake.
- Nutrient-dense: Our Protein Shake is packed with essential vitamins and minerals, inclu

5